# CNN-GRU : amélioration par des entrées physiquement motivées

Le CNN-GRU de référence reçoit 2 canaux : la température normalisée et l'excitation. Il doit séparer seul la réponse libre (météo, état initial) de la réponse au chauffage, alors que l'analyse avec et sans excitation a montré que ces deux réponses portent des informations différentes.

La version améliorée ajoute 3 canaux qui font une partie de ce travail :
- **T − T_libre** : la température moins la réponse libre moyenne (calculée sur le train), approximation de la réponse au chauffage h ∗ U ;
- **T − T(0)** : l'élévation depuis le départ, indépendante du niveau initial ;
- **le temps t / 8 h** : la position dans la séquence, donc dans la météo commune, qu'une convolution ne connaît pas d'elle-même.

Une seule modification est testée : les entrées. Codage des cibles (contraint), architecture et réglages d'entraînement sont identiques à la référence.

## 1. Configuration et chargement des données

In [ ]:
import copy
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# --- Chemins (Kaggle) ---
DATA = Path("/kaggle/input/datasets/jiihan/coenergy")   # données préparées + splits.pt + .py
OUT = Path("/kaggle/working")                           # modèles et historiques sauvegardés

# --- Réglages communs (identiques à la référence) ---
SEED = 42
N_EPOCHS = 10
LR = 5e-4
BATCH_SIZE = 256
N_TRAIN = None          # None = tout le train (800 000), comme le modèle de référence sauvegardé
P_MAX = 7000.0
ENCODING = "contraint"

# Référence : True = recharger le CNN-GRU déjà entraîné (cnngru_contraint_best.pt), sans le réentraîner.
# Valable seulement si N_TRAIN = None (mêmes conditions que lors de son entraînement).
LOAD_REFERENCE = True
REFERENCE_DIR = OUT     # dossier contenant cnngru_contraint_best.pt et cnngru_contraint_history.pt

sys.path.append(str(DATA))
from data_utils import load_data

T, U, Y, targets = load_data(DATA)          # T en °C, U en W (float32), Y (N, 6)
SPLITS = torch.load(DATA / "splits.pt")     # découpage stratifié déjà calculé

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil :", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print(T.shape, U.shape, Y.shape)

In [ ]:
targets = [str(t) for t in targets]
col = {name: i for i, name in enumerate(targets)}
N, L = T.shape

NO_EXC = U.sum(1) == 0                      # simulations sans excitation
ALWAYS_ON = (U == P_MAX).all(1)             # chauffage permanent

for name, idx in SPLITS.items():
    print(f"{name:6s} {len(idx):7d} simulations | sans excitation {NO_EXC[idx].float().mean().item():.2%}"
          f" | toujours allumé {ALWAYS_ON[idx].float().mean().item():.2%}")

## 2. Normalisation des entrées et canaux physiques

In [ ]:
# Température : une seule moyenne et un seul écart-type, calculés sur le train
T_SD, T_MU = torch.std_mean(T[SPLITS["train"][:100_000]])
T_MU, T_SD = T_MU.item(), T_SD.item()

# Réponse libre moyenne : moyenne des simulations SANS excitation du train
idx_free = SPLITS["train"][NO_EXC[SPLITS["train"]]]
T_FREE = T[idx_free].mean(0)                         # (480,)
TIME = torch.linspace(0, 1, L)                       # position dans la séquence, de 0 à 1

# Écarts-types des nouveaux canaux, pour les mettre à la même échelle
s = T[SPLITS["train"][:50_000]]
D_FREE_SD = (s - T_FREE).std().item()
D_T0_SD = (s - s[:, :1]).std().item()
del s

print(f"T : moyenne = {T_MU:.3f} °C, écart-type = {T_SD:.3f} °C")
print(f"Réponse libre estimée sur {len(idx_free)} simulations")
print(f"sd(T - T_libre) = {D_FREE_SD:.3f} °C | sd(T - T0) = {D_T0_SD:.3f} °C")

plt.figure(figsize=(9, 3))
plt.plot(TIME.numpy() * L / 60, T_FREE.numpy())
plt.xlabel("Temps (h)")
plt.ylabel("T_libre (°C)")
plt.title("Réponse libre moyenne (simulations sans excitation du train)")
plt.show()

## 3. Codage des cibles

Codage contraint : HLC_mit, HLC_ext, log τ_n, log(τ_∞ − τ_n), log(τ_b − τ_∞). HLC = HLC_mit + HLC_ext et τ_n < τ_∞ < τ_b sont garantis par construction. Le codage direct est conservé pour référence.

In [ ]:
def encode_contraint(y: torch.Tensor) -> torch.Tensor:
    """6 indicateurs -> HLC_mit, HLC_ext, log tau_n, log(tau_inf - tau_n), log(tau_b - tau_inf)."""
    tb, tn, ti = y[:, col["tau_b"]], y[:, col["tau_n"]], y[:, col["tau_inf"]]
    return torch.stack([y[:, col["HLC_mit"]], y[:, col["HLC_ext"]],
                        tn.log(), (ti - tn).log(), (tb - ti).log()], dim=1)


def decode_contraint(z: torch.Tensor) -> torch.Tensor:
    """5 valeurs -> 6 indicateurs : HLC = somme, et tau_n < tau_inf < tau_b garantis."""
    tn = z[:, 2].exp()
    ti = tn + z[:, 3].exp()
    tb = ti + z[:, 4].exp()
    y = torch.empty(len(z), 6, dtype=z.dtype)
    y[:, col["HLC_mit"]], y[:, col["HLC_ext"]] = z[:, 0], z[:, 1]
    y[:, col["HLC"]] = z[:, 0] + z[:, 1]
    y[:, col["tau_n"]], y[:, col["tau_inf"]], y[:, col["tau_b"]] = tn, ti, tb
    return y


def encode_direct(y: torch.Tensor) -> torch.Tensor:
    """6 indicateurs -> HLC_mit, HLC_ext, tau_b, tau_n, tau_inf."""
    return torch.stack([y[:, col["HLC_mit"]], y[:, col["HLC_ext"]],
                        y[:, col["tau_b"]], y[:, col["tau_n"]], y[:, col["tau_inf"]]], dim=1)


def decode_direct(z: torch.Tensor) -> torch.Tensor:
    """5 valeurs -> 6 indicateurs : HLC = somme garanti, ordre des tau non garanti."""
    y = torch.empty(len(z), 6, dtype=z.dtype)
    y[:, col["HLC_mit"]], y[:, col["HLC_ext"]] = z[:, 0], z[:, 1]
    y[:, col["HLC"]] = z[:, 0] + z[:, 1]
    y[:, col["tau_b"]], y[:, col["tau_n"]], y[:, col["tau_inf"]] = z[:, 2], z[:, 3], z[:, 4]
    return y


ENCODINGS = {"contraint": (encode_contraint, decode_contraint),
             "direct": (encode_direct, decode_direct)}

for name, (enc, dec) in ENCODINGS.items():
    y = Y[:1000].double()
    assert torch.allclose(dec(enc(y)), y, rtol=1e-6), name
print("Codages vérifiés.")

## 4. Dataset et métriques

In [ ]:
INPUTS = {"base": 2, "physique": 5}         # nombre de canaux d'entrée de chaque version


class MyDataset(Dataset):
    """Renvoie une simulation : entrée (C, L) et cible standardisée (5,).

    inputs = "base"     : [T normalisé, U / P_MAX]
    inputs = "physique" : [T normalisé, U / P_MAX, T - T_libre, T - T(0), temps]
    """

    def __init__(self, idx: torch.Tensor, z_std: torch.Tensor, inputs: str = "base",
                 noise_c: float = 0.0) -> None:
        self.idx = idx
        self.z_std = z_std
        self.inputs = inputs
        self.noise_c = noise_c

    def __len__(self) -> int:
        return len(self.idx)

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        sim = self.idx[i]
        t = T[sim]
        if self.noise_c > 0:
            t = t + self.noise_c * torch.randn_like(t)
        channels = [(t - T_MU) / T_SD, U[sim] / P_MAX]
        if self.inputs == "physique":
            channels += [(t - T_FREE) / D_FREE_SD,      # réponse au chauffage (approx.)
                         (t - t[0]) / D_T0_SD,          # élévation depuis le départ
                         TIME]                          # position dans la séquence
        return torch.stack(channels), self.z_std[sim]


def regression_metrics(y_true: torch.Tensor, y_pred: torch.Tensor) -> dict:
    """RMSE, MAE, P95, MAPE et R² par indicateur, en unités physiques."""
    out = {}
    for name in targets:
        t, p = y_true[:, col[name]].double(), y_pred[:, col[name]].double()
        err = p - t
        out[name] = {"RMSE": err.pow(2).mean().sqrt().item(),
                     "MAE": err.abs().mean().item(),
                     "P95": err.abs().quantile(0.95).item(),
                     "MAPE": 100 * (err / t).abs().mean().item(),
                     "R2": (1 - err.pow(2).sum() / (t - t.mean()).pow(2).sum()).item()}
    return out


RESULTS = {}


def evaluate(name: str, y_pred: torch.Tensor, split: str = "val") -> None:
    """Évalue un modèle globalement, avec et sans excitation, et enregistre le résultat."""
    idx = SPLITS[split]
    y_true, no_exc = Y[idx], NO_EXC[idx]
    RESULTS[name] = {g: regression_metrics(y_true[m], y_pred[m])
                     for g, m in [("tout", torch.ones_like(no_exc)),
                                  ("avec excitation", ~no_exc), ("sans excitation", no_exc)]}
    print(f"=== {name} ({split}) ===")
    print(f"{'':12s}{'RMSE':>9s}{'MAE':>9s}{'P95':>9s}{'MAPE %':>9s}{'R²':>8s}{'R² exc.':>9s}")
    for t in targets:
        m, me = RESULTS[name]["tout"][t], RESULTS[name]["avec excitation"][t]
        print(f"{t:12s}{m['RMSE']:9.2f}{m['MAE']:9.2f}{m['P95']:9.2f}"
              f"{m['MAPE']:9.2f}{m['R2']:8.3f}{me['R2']:9.3f}")


evaluate("Moyenne", Y[SPLITS["train"]].mean(0).expand(len(SPLITS["val"]), -1))

for inputs in INPUTS:
    x, _ = MyDataset(SPLITS["train"][:4], torch.zeros(N, 5), inputs)[0]
    print(f"Entrée « {inputs} » : {tuple(x.shape)}")

## 5. Le modèle CNN-GRU

In [ ]:
class CNNGRURegressor(nn.Module):
    """CNN-GRU : séquence (B, C, L) -> 5 sorties standardisées."""

    def __init__(self, in_channels: int = 2, channels: int = 64, kernel: int = 5, n_conv: int = 3,
                 hidden: int = 64, n_gru_layers: int = 2, dropout: float = 0.1, n_out: int = 5) -> None:
        super().__init__()
        layers, c_in = [], in_channels
        for _ in range(n_conv):
            layers += [nn.Conv1d(c_in, channels, kernel, padding=kernel // 2),
                       nn.BatchNorm1d(channels), nn.GELU(), nn.MaxPool1d(2)]
            c_in = channels
        self.cnn = nn.Sequential(*layers)                      # (B, channels, L / 2^n_conv)
        self.gru = nn.GRU(input_size=channels, hidden_size=hidden, num_layers=n_gru_layers,
                          batch_first=True, bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(2 * hidden, 64), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(64, n_out))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.cnn(x).transpose(1, 2)                         # (B, 60, channels)
        _, h_n = self.gru(h)                                    # (n_couches * 2, B, hidden)
        return self.head(torch.cat([h_n[-2], h_n[-1]], dim=1))  # états finaux aller + retour


def weights_init(layer_in):
    """Initialisation de He pour les convolutions et les couches linéaires (le GRU garde la sienne)."""
    if isinstance(layer_in, (nn.Linear, nn.Conv1d)):
        nn.init.kaiming_uniform_(layer_in.weight)
        layer_in.bias.data.fill_(0.0)


for inputs, c in INPUTS.items():
    m = CNNGRURegressor(in_channels=c)
    print(f"CNN-GRU « {inputs} » ({c} canaux) : {sum(p.numel() for p in m.parameters()):,} paramètres")
del m

## 6. Entraînement et chargement

`run_experiment` entraîne un nouveau modèle ; `load_experiment` recharge un modèle déjà entraîné et l'évalue. Les fichiers de la référence gardent leurs noms d'origine (`cnngru_contraint_…`), ceux de la version améliorée commencent par `cnngru5_`.

In [ ]:
def file_prefix(inputs: str) -> str:
    return "cnngru" if inputs == "base" else "cnngru5"


def targets_std(encoding: str):
    """Cibles standardisées pour un codage (statistiques du train)."""
    encode, decode = ENCODINGS[encoding]
    Z = encode(Y.double()).float()
    z_mu, z_sd = Z[SPLITS["train"]].mean(0), Z[SPLITS["train"]].std(0)
    return (Z - z_mu) / z_sd, z_mu, z_sd, decode


@torch.no_grad()
def predict(model, loader, z_mu, z_sd, decode) -> torch.Tensor:
    """Prédictions en unités physiques (n, 6). Le loader ne doit PAS être mélangé."""
    model.eval()
    z = torch.cat([model(x.to(DEVICE)).float().cpu() for x, _ in loader])
    return decode((z * z_sd + z_mu).double()).float()


def eval_loader(split: str, z_std: torch.Tensor, inputs: str) -> DataLoader:
    return DataLoader(MyDataset(SPLITS[split], z_std, inputs), batch_size=1024, shuffle=False,
                      num_workers=2, pin_memory=True)


def run_experiment(inputs: str, encoding: str = ENCODING) -> dict:
    """Entraîne et évalue un CNN-GRU pour une version des entrées et un codage des cibles."""
    torch.manual_seed(SEED)
    z_std, z_mu, z_sd, decode = targets_std(encoding)
    prefix = f"{file_prefix(inputs)}_{encoding}"

    train_idx = SPLITS["train"] if N_TRAIN is None else SPLITS["train"][:N_TRAIN]
    train_loader = DataLoader(MyDataset(train_idx, z_std, inputs), batch_size=BATCH_SIZE, shuffle=True,
                              drop_last=True, num_workers=2, pin_memory=True)
    val_loader = eval_loader("val", z_std, inputs)

    model = CNNGRURegressor(in_channels=INPUTS[inputs])
    model.apply(weights_init)
    model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    best_val, best_state, history = float("inf"), None, []
    for epoch in range(N_EPOCHS):
        start = time.time()
        model.train()
        epoch_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
        epoch_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                val_loss += criterion(model(x_batch.to(DEVICE)), y_batch.to(DEVICE)).item()
        val_loss /= len(val_loader)

        print(f"[{inputs}] Epoch {epoch:3d}, train loss {epoch_loss:.4f}, val loss {val_loss:.4f}, "
              f"lr {scheduler.get_last_lr()[0]:.1e}, {time.time() - start:.0f} s")
        history.append({"epoch": epoch + 1, "train": epoch_loss, "val": val_loss})
        torch.save(history, OUT / f"{prefix}_history.pt")

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, OUT / f"{prefix}_best.pt")
        scheduler.step()

    model.load_state_dict(best_state)
    y_pred = predict(model, val_loader, z_mu, z_sd, decode)
    evaluate(f"CNN-GRU {inputs}", y_pred)
    return {"model": model, "history": history, "y_pred": y_pred}


def load_experiment(inputs: str, folder: Path, encoding: str = ENCODING) -> dict:
    """Recharge un CNN-GRU déjà entraîné et l'évalue sur la validation."""
    z_std, z_mu, z_sd, decode = targets_std(encoding)
    prefix = f"{file_prefix(inputs)}_{encoding}"
    model = CNNGRURegressor(in_channels=INPUTS[inputs]).to(DEVICE)
    model.load_state_dict(torch.load(folder / f"{prefix}_best.pt", map_location=DEVICE))
    hist_file = folder / f"{prefix}_history.pt"
    history = torch.load(hist_file) if hist_file.exists() else []
    y_pred = predict(model, eval_loader("val", z_std, inputs), z_mu, z_sd, decode)
    evaluate(f"CNN-GRU {inputs}", y_pred)
    return {"model": model, "history": history, "y_pred": y_pred}

## 7. Référence : CNN-GRU à 2 canaux

In [ ]:
if LOAD_REFERENCE:
    assert N_TRAIN is None, "la référence sauvegardée a été entraînée sur tout le train"
    run_base = load_experiment("base", REFERENCE_DIR)
else:
    run_base = run_experiment("base")

## 8. Version améliorée : CNN-GRU à 5 canaux

In [ ]:
run_phys = run_experiment("physique")

## 9. Comparaison

In [ ]:
# Courbes d'apprentissage : même codage des cibles, les pertes sont donc comparables
fig, ax = plt.subplots(figsize=(9, 4))
for name, run, color in [("référence (2 canaux)", run_base, "#2a78d6"),
                         ("améliorée (5 canaux)", run_phys, "#eb6834")]:
    if run["history"]:
        ep = [h["epoch"] for h in run["history"]]
        ax.plot(ep, [h["train"] for h in run["history"]], color=color, ls="--", lw=1.5, label=f"{name}, entraînement")
        ax.plot(ep, [h["val"] for h in run["history"]], color=color, lw=2, marker="o", ms=4, label=f"{name}, validation")
ax.set(xlabel="Époque", ylabel="MSE (cibles standardisées)", title="CNN-GRU : effet des canaux physiques")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# R² par indicateur, en unités physiques (validation)
models = ["Moyenne", "CNN-GRU base", "CNN-GRU physique"]
for group in ["tout", "avec excitation", "sans excitation"]:
    print(f"\nR² — {group}")
    print(f"{'':20s}" + "".join(f"{t:>10s}" for t in targets))
    for name in models:
        print(f"{name:20s}" + "".join(f"{RESULTS[name][group][t]['R2']:10.3f}" for t in targets))

print(f"\nGain de R² (physique − base), tout :")
print(f"{'':20s}" + "".join(f"{RESULTS['CNN-GRU physique']['tout'][t]['R2'] - RESULTS['CNN-GRU base']['tout'][t]['R2']:+10.3f}"
                            for t in targets))

print(f"\nMAE (W/K pour les HLC, h pour les tau) — tout")
print(f"{'':20s}" + "".join(f"{t:>10s}" for t in targets))
for name in models:
    print(f"{name:20s}" + "".join(f"{RESULTS[name]['tout'][t]['MAE']:10.2f}" for t in targets))

In [ ]:
# Contraintes physiques
for name, run in [("base", run_base), ("physique", run_phys)]:
    y = run["y_pred"]
    somme = torch.allclose(y[:, col["HLC"]], y[:, col["HLC_mit"]] + y[:, col["HLC_ext"]])
    ordre = (y[:, col["tau_n"]] < y[:, col["tau_inf"]]) & (y[:, col["tau_inf"]] < y[:, col["tau_b"]])
    print(f"{name:9s} | HLC = HLC_mit + HLC_ext : {somme} | violations de l'ordre des tau : {(~ordre).float().mean().item():.2%}")

In [ ]:
# Graphique de synthèse : R² par indicateur, référence contre version améliorée
x = np.arange(len(targets))
width = 0.38
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
for ax, group in zip(axes, ["avec excitation", "sans excitation"]):
    for k, (name, color) in enumerate([("CNN-GRU base", "#2a78d6"), ("CNN-GRU physique", "#eb6834")]):
        vals = [RESULTS[name][group][t]["R2"] for t in targets]
        bars = ax.bar(x + (k - 0.5) * (width + 0.03), vals, width, color=color, label=name)
        ax.bar_label(bars, fmt="%.2f", fontsize=8)
    ax.set_xticks(x, targets)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"Simulations {group}")
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("R² (validation)")
axes[0].legend()
plt.tight_layout()
plt.show()

## 10. Évaluation finale sur le test

À exécuter **une seule fois**, pour le modèle retenu, après avoir comparé les versions sur la validation.

In [ ]:
FINAL = "physique"          # "base" ou "physique", selon la comparaison ci-dessus
RUN_TEST = False            # passer à True pour lancer l'évaluation finale

if RUN_TEST:
    z_std, z_mu, z_sd, decode = targets_std(ENCODING)
    final_model = run_phys["model"] if FINAL == "physique" else run_base["model"]
    y_test = predict(final_model, eval_loader("test", z_std, FINAL), z_mu, z_sd, decode)
    evaluate(f"CNN-GRU {FINAL} (test)", y_test, split="test")